# 11 — Manual, horizon-aware early stopping (P3 retry)

**Goal:** P3's hyperparameter random search (`notebooks/10_hyperparameter_tuning.ipynb`)
found a candidate that clearly beat the current defaults on this project's own 5-seed
proxy (0.7117 -> 0.7082, ~0.49%, 4/5 seeds) but scored **worse** on a real Zindi
submission (0.758156 vs. the current best 0.755129) — a genuine proxy-real inversion on
the hyperparameter axis. The working hypothesis (README, RESOURCES.md, 2026-09-05,
until now unverified) was that `HistGradientBoostingRegressor`'s built-in
`early_stopping` decides how many boosting rounds to run using an internal validation
split that has no relationship to this project's horizon-matched/mask-aware validation
structure — so the exact stopping point a candidate lands on, especially at very low
`learning_rate` where this matters most, may not transfer to Test.csv's real deployment
distribution.

**This session, that hypothesis was checked against the scikit-learn 1.7.2 source**
(`sklearn/ensemble/_hist_gradient_boosting/gradient_boosting.py`), not just assumed:
- Lines ~672-679: when `early_stopping` is on and no validation data is supplied,
  `fit()` calls `sklearn.model_selection.train_test_split(X, y, test_size=validation_fraction,
  random_state=...)` with no `shuffle=False` — i.e. a plain **random, IID** split of
  whatever rows it's given, with no awareness of time or forecast horizon. Confirms the
  hypothesis's mechanism exactly.
- Lines 511-551: this sklearn version's `fit()` signature accepts `X_val`, `y_val`,
  `sample_weight_val` (`versionadded:: 1.7`) — "Additional sample of features/target
  for validation **used in early stopping**". Lines 577-596 confirm: if these are
  passed, the internal random split is skipped entirely and the supplied validation
  data drives the stopping decision instead. **This notebook requires scikit-learn
  >= 1.7** for that API — check/upgrade in the next cell.

**Sources for the fix's design:**
- The scikit-learn `HistGradientBoostingRegressor.fit()` docstring above (this
  session's finding) — the `X_val`/`y_val` mechanism itself.
- This project's own already-established chronological-split rationale (scikit-learn's
  `TimeSeriesSplit` docs, cited in `RESOURCES.md`, implemented as
  `src/evaluate.py::time_train_val_split`) — applied a second time, recursively, to
  carve the early-stopping holdout out of the *fit* portion only. No new source is
  needed for this part: it is the same "don't validate a forecasting task on a random
  slice" principle already governing every split in this project, just applied one
  level deeper (to the model's internal stopping decision, not only the outer
  fit/val split).
- Same search space, same seed, same methodology as
  `notebooks/10_hyperparameter_tuning.ipynb` (Bergstra & Bengio, JMLR 2012; `hands-on-ml`
  ch07; `real-world-ml` ch04) — deliberately unchanged, so any difference in outcome is
  attributable to the early-stopping fix alone, not a different search.

**Runs locally, not on Kaggle**: the user has a stronger local CPU than Kaggle's free
tier, and `HistGradientBoostingRegressor` is CPU-only in scikit-learn (no GPU support
at all) — Kaggle's GPU accelerator would not have helped this specific workload anyway.
To keep the machine usable for other work while this runs, every model-fitting cell is
wrapped in a `threadpoolctl.threadpool_limits` cap (see the next cell) — scikit-learn's
own documented mechanism for throttling the OpenMP/BLAS thread pools its Cython code
uses internally (`HistGradientBoostingRegressor` has no `n_jobs` constructor argument;
this is the correct knob, not an environment variable set mid-session, which OpenMP
runtimes don't reliably re-read after first use).

**Imports directly from `src`** (`config`, `data`, `evaluate`, `features`, `model`,
`train.get_feature_cols`) exactly like notebooks 08-10 — this notebook has no reason to
duplicate that logic now that it runs against the local repo rather than a Kaggle
kernel with no filesystem access to it. (An earlier version of this notebook, written
before the user decided to run locally instead of on Kaggle, inlined copies of those
modules for portability; that inlining was removed once it was no longer needed, per
this project's own DRY convention — see `structuring-ml-projects`.)


In [1]:
import sklearn
print("scikit-learn:", sklearn.__version__)
major, minor = (int(x) for x in sklearn.__version__.split(".")[:2])
assert (major, minor) >= (1, 7), (
    "This notebook needs scikit-learn >= 1.7 for HistGradientBoostingRegressor's "
    "X_val/y_val early-stopping API. `pip install -U scikit-learn` if this fails."
)


scikit-learn: 1.7.2


## CPU throttle

`N_THREADS` caps every OpenMP/BLAS-backed fit below at this many threads for the rest
of the kernel session, leaving the remaining logical CPUs free for other work. Default
below leaves roughly half of a 22-core machine free — **edit `N_THREADS` directly** to
taste (there is no other setting to change; every cell reuses this one cap). Keeping
`_thread_limiter` alive as a module-level variable is what makes the cap persist across
cells — `threadpool_limits` used as a bare context manager would restore full
parallelism the instant it exits. Call `_thread_limiter.unregister()` to lift the cap
early if wanted.


In [2]:
import os

import threadpoolctl

N_THREADS = max(1, os.cpu_count() // 2)  # edit to taste; os.cpu_count() = logical CPUs
_thread_limiter = threadpoolctl.threadpool_limits(limits=N_THREADS)
print(f"Capped native thread pools (OpenMP/BLAS) at {N_THREADS} of {os.cpu_count()} logical CPUs.")


Capped native thread pools (OpenMP/BLAS) at 11 of 22 logical CPUs.


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from src import config, data, evaluate, features, model
from src.train import get_feature_cols

pd.set_option("display.width", 120)


## Load data

Fast (seconds) — defines `raw_train`, `target_horizons`, `masked_month_fraction`,
`masked_row_fraction`, all needed by every cell below. **Always run this one** — there
is no way to skip it, unlike the optional slow sanity check in the next cell.


In [4]:
raw_train, test, _ = data.load_raw_data()
target_horizons = evaluate.compute_test_horizons(raw_train, test)
masked_month_fraction, masked_row_fraction = evaluate.measure_masking_pattern(test)
print(f"target_horizons: {sorted(target_horizons)}")
print(f"masked_month_fraction={masked_month_fraction:.4f} masked_row_fraction={masked_row_fraction:.4f}")


target_horizons: [1, 5, 6, 7, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 35, 39, 40]
masked_month_fraction=0.6667 masked_row_fraction=0.9978


## Optional, slow: reproduce the current best proxy before trusting anything new

Per `structuring-ml-projects`' rule of recomputing the baseline in the same run before
any comparison: this reproduces ~0.7126 mean RMSE over 5 seeds (the number
`src/train.py` reports) — takes several minutes (5 full fit/feature-build cycles).

**Already confirmed this session, twice** (output below is real, captured when this
exact code — first as an inlined standalone copy, then again unchanged after switching
to these `src` imports — was actually run against local Train.csv/Test.csv: 0.7117
both times, well within the project's known 0.69-0.73 seed-to-seed range). Unlike the
data-loading cell above, **this one is safe to skip** — it defines no variable any
later cell needs, it only prints a confirmation. Skip it (or "Run All Above" the
search cells further down without running this one) if you don't want to spend the
several minutes it takes; rerun it only if `src/features.py`/`src/evaluate.py`/
`src/model.py` change and you want to re-confirm.


In [5]:
baseline_rmses = []
for seed in range(5):
    fit_s, val_s = evaluate.mask_augmented_horizon_matched_split(
        raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
        fit_seed=seed, val_seed=seed + 100,
    )
    cols = get_feature_cols(fit_s)
    Xf = features.select_base_features(fit_s, cols)
    yf = fit_s[config.TARGET_COL].to_numpy()
    Xv = features.select_base_features(val_s, cols)
    yv = val_s[config.TARGET_COL].to_numpy()
    m = model.make_baseline_model()
    m.fit(Xf, yf)
    baseline_rmses.append(evaluate.rmse(yv, model.predict(m, Xv)))

print(f"Reproduced baseline: mean={np.mean(baseline_rmses):.4f} "
      f"range=({min(baseline_rmses):.4f}, {max(baseline_rmses):.4f})  {baseline_rmses}")
print("Expected (src/train.py, local run): mean ~= 0.7126")


Reproduced baseline: mean=0.7117 range=(0.6925, 0.7276)  [0.7276082633085404, 0.7222508079865849, 0.7160058108161343, 0.6925121448250914, 0.700238667633396]
Expected (src/train.py, local run): mean ~= 0.7126


## Search space (unchanged from P3 / notebook 10)

Deliberately identical to `notebooks/10_hyperparameter_tuning.ipynb`'s round-1 space —
same seed, same distributions — so this run isolates the effect of the early-stopping
fix. `max_iter` stays fixed at a generous ceiling; early stopping (now horizon-aware)
picks the actual count.


In [ ]:
def sample_configs(n, seed):
    rng = np.random.RandomState(seed)
    configs = []
    for _ in range(n):
        configs.append(dict(
            learning_rate=float(10 ** rng.uniform(np.log10(0.01), np.log10(0.3))),
            max_depth=int(rng.randint(3, 13)),
            min_samples_leaf=int(10 ** rng.uniform(np.log10(10), np.log10(200))),
            l2_regularization=float(10 ** rng.uniform(np.log10(0.01), np.log10(10))),
            max_leaf_nodes=int(rng.randint(15, 128)),
        ))
    return configs


N_CANDIDATES = 25
candidates = sample_configs(N_CANDIDATES, seed=config.RANDOM_STATE)
print(f"{N_CANDIDATES} candidates sampled.")


## Manual, horizon-aware early stopping

`fit_df` (from `evaluate.mask_augmented_horizon_matched_split`) is already fully
featurized and already carries realistic masking-augmented rows scattered across its
months. Carving its temporally-last slice off with `evaluate.time_train_val_split` —
the same chronological-split function already used everywhere else in this project —
gives an early-stopping holdout that is disjoint from both the training rows and the
outer `val_df` used to score/rank candidates, with no new feature-engineering code
required.

The imputer is fit on the inner-fit rows only and applied to all three slices by hand
(not via `Pipeline`, since `Pipeline.fit()` does not transform `fit_params` like
`gbr__X_val` through earlier steps automatically) — this keeps `SimpleImputer(median)`
consistent with `model.make_baseline_model()`'s choice without silently changing it.


In [ ]:
def make_gbr(params):
    return HistGradientBoostingRegressor(
        loss="squared_error",
        max_iter=1000,
        early_stopping=True,
        random_state=config.RANDOM_STATE,
        **params,
    )


def fit_with_manual_early_stopping(params, fit_df, feature_cols, early_stop_fraction=0.15):
    fit_inner_df, early_stop_df = evaluate.time_train_val_split(fit_df, val_fraction=early_stop_fraction)

    X_fit_inner = features.select_base_features(fit_inner_df, feature_cols)
    y_fit_inner = fit_inner_df[config.TARGET_COL].to_numpy()
    X_early_stop = features.select_base_features(early_stop_df, feature_cols)
    y_early_stop = early_stop_df[config.TARGET_COL].to_numpy()

    imputer = SimpleImputer(strategy="median").fit(X_fit_inner)
    X_fit_inner_imp = imputer.transform(X_fit_inner)
    X_early_stop_imp = imputer.transform(X_early_stop)

    gbr = make_gbr(params)
    gbr.fit(X_fit_inner_imp, y_fit_inner, X_val=X_early_stop_imp, y_val=y_early_stop)
    return gbr, imputer, gbr.n_iter_


def predict_with(gbr, imputer, X):
    return gbr.predict(imputer.transform(X))


## Screening pass (single seed) — manual vs. default early stopping

Compares, on the exact same fit/val split: (a) the current defaults with sklearn's
default (random-holdout) early stopping — reproduces notebook 10's screening baseline
— against (b) the current defaults AND every candidate with the manual, horizon-aware
early stopping fix. `n_iter_` (actual boosting rounds used) is logged for each, since a
large shift there is the whole point of this fix.


In [6]:
fit_df, val_df = evaluate.mask_augmented_horizon_matched_split(
    raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
    fit_seed=0, val_seed=100,
)
feature_cols = get_feature_cols(fit_df)
X_val = features.select_base_features(val_df, feature_cols)
y_val = val_df[config.TARGET_COL].to_numpy()

# (a) Reproduce notebook 10's default-early-stopping baseline on this split.
X_fit_full = features.select_base_features(fit_df, feature_cols)
y_fit_full = fit_df[config.TARGET_COL].to_numpy()
default_es_model = model.make_baseline_model()
default_es_model.fit(X_fit_full, y_fit_full)
default_es_rmse = evaluate.rmse(y_val, model.predict(default_es_model, X_val))
print(f"Current defaults, DEFAULT (random-holdout) early stopping: rmse={default_es_rmse:.4f}")

# (b) Current defaults with the manual, horizon-aware fix.
current_defaults = dict(learning_rate=0.05, max_depth=8, min_samples_leaf=50, l2_regularization=1.0)
gbr, imputer, n_iter = fit_with_manual_early_stopping(current_defaults, fit_df, feature_cols)
manual_es_rmse = evaluate.rmse(y_val, predict_with(gbr, imputer, X_val))
print(f"Current defaults, MANUAL horizon-aware early stopping: rmse={manual_es_rmse:.4f} (n_iter_={n_iter})")

# (c) Every P3 candidate, manual early stopping only (this is the fix under test).
screening_results = []
for i, params in enumerate(candidates):
    gbr, imputer, n_iter = fit_with_manual_early_stopping(params, fit_df, feature_cols)
    r = evaluate.rmse(y_val, predict_with(gbr, imputer, X_val))
    screening_results.append({**params, "rmse": r, "n_iter": n_iter})
    print(f"[{i+1}/{N_CANDIDATES}] rmse={r:.4f} n_iter={n_iter} params={params}")

screening_df = pd.DataFrame(screening_results).sort_values("rmse").reset_index(drop=True)
screening_df


Current defaults, DEFAULT (random-holdout) early stopping: rmse=0.7276
Current defaults, MANUAL horizon-aware early stopping: rmse=0.7362 (n_iter_=176)
[1/25] rmse=0.7296 n_iter=157 params={'learning_rate': 0.03574712922600243, 'max_depth': 10, 'min_samples_leaf': 60, 'l2_regularization': 0.029380279387035357, 'max_leaf_nodes': 97}
[2/25] rmse=0.7311 n_iter=603 params={'learning_rate': 0.014049959523783894, 'max_depth': 10, 'min_samples_leaf': 27, 'l2_regularization': 0.026828750938254375, 'max_leaf_nodes': 17}
[3/25] rmse=0.7312 n_iter=830 params={'learning_rate': 0.010725209743171996, 'max_depth': 4, 'min_samples_leaf': 86, 'l2_regularization': 6.541210527692732, 'max_leaf_nodes': 16}
[4/25] rmse=0.7302 n_iter=227 params={'learning_rate': 0.018559980846490572, 'max_depth': 7, 'min_samples_leaf': 63, 'l2_regularization': 0.6838478430964042, 'max_leaf_nodes': 122}
[5/25] rmse=0.7319 n_iter=797 params={'learning_rate': 0.010815983055225084, 'max_depth': 12, 'min_samples_leaf': 11, 'l2_r

## Confirmation pass (5-seed mean) for the top candidates

Same discipline as notebook 10: don't trust a single-seed screening result. Re-evaluate
the top 3 screened candidates AND the current defaults (both early-stopping variants)
over the full 5-seed mean before drawing any conclusion.


In [7]:
def five_seed_mean_rmse(params_or_none, manual_early_stopping):
    rmses = []
    n_iters = []
    for seed in range(5):
        fit_s, val_s = evaluate.mask_augmented_horizon_matched_split(
            raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
            fit_seed=seed, val_seed=seed + 100,
        )
        cols = get_feature_cols(fit_s)
        Xv = features.select_base_features(val_s, cols)
        yv = val_s[config.TARGET_COL].to_numpy()

        if manual_early_stopping:
            params = current_defaults if params_or_none is None else params_or_none
            gbr, imputer, n_iter = fit_with_manual_early_stopping(params, fit_s, cols)
            rmses.append(evaluate.rmse(yv, predict_with(gbr, imputer, Xv)))
            n_iters.append(n_iter)
        else:
            Xf = features.select_base_features(fit_s, cols)
            yf = fit_s[config.TARGET_COL].to_numpy()
            m = model.make_baseline_model() if params_or_none is None else Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("gbr", make_gbr(params_or_none)),
            ])
            m.fit(Xf, yf)
            rmses.append(evaluate.rmse(yv, model.predict(m, Xv)))
            n_iters.append(m.named_steps["gbr"].n_iter_)
    return rmses, n_iters


top3 = screening_df.head(3).to_dict("records")
param_cols = ["learning_rate", "max_depth", "min_samples_leaf", "l2_regularization", "max_leaf_nodes"]

confirmation = {}
confirmation["current_defaults__default_es"] = five_seed_mean_rmse(None, manual_early_stopping=False)
confirmation["current_defaults__manual_es"] = five_seed_mean_rmse(None, manual_early_stopping=True)
for i, row in enumerate(top3):
    params = {k: row[k] for k in param_cols}
    confirmation[f"candidate_{i+1}__manual_es"] = five_seed_mean_rmse(params, manual_early_stopping=True)

for name, (rmses, n_iters) in confirmation.items():
    print(f"{name}: mean_rmse={np.mean(rmses):.4f} range=({min(rmses):.4f}, {max(rmses):.4f}) "
          f"mean_n_iter={np.mean(n_iters):.0f}  rmses={[round(r, 4) for r in rmses]}")


current_defaults__default_es: mean_rmse=0.7117 range=(0.6925, 0.7276) mean_n_iter=300  rmses=[0.7276, 0.7223, 0.716, 0.6925, 0.7002]
current_defaults__manual_es: mean_rmse=0.7176 range=(0.6999, 0.7362) mean_n_iter=195  rmses=[0.7362, 0.7249, 0.7207, 0.6999, 0.7062]
candidate_1__manual_es: mean_rmse=0.7163 range=(0.7036, 0.7263) mean_n_iter=173  rmses=[0.7259, 0.7263, 0.72, 0.7057, 0.7036]
candidate_2__manual_es: mean_rmse=0.7164 range=(0.7012, 0.7287) mean_n_iter=239  rmses=[0.7287, 0.7282, 0.7193, 0.7012, 0.7046]
candidate_3__manual_es: mean_rmse=0.7187 range=(0.7053, 0.7298) mean_n_iter=123  rmses=[0.7289, 0.7298, 0.7228, 0.7053, 0.7067]


## Independent review (Opus) found a confound in the comparison above

The 5-seed confirmation above shows `current_defaults__manual_es` losing to
`current_defaults__default_es` in 5/5 seeds, uniformly. Before accepting "the
early-stopping mechanism is refuted" from that alone, an independent review (this
project's established practice for a key decision point, per the 2026-09-04 Opus
review precedent in README.md) was asked to check the methodology from scratch. It
found:

**The comparison is confounded, not clean.** `evaluate.time_train_val_split` is
*chronological* — so `fit_with_manual_early_stopping`'s inner split
(`val_fraction=0.15`) removes the temporally-**last** 15% of `fit_df`'s months as the
early-stopping holdout, and the model never trains on them at all. The DEFAULT-early-
stopping arm, by contrast, holds out sklearn's internal random 10% of *rows* spread
evenly across every month — it still trains on the full time range. So the manual-ES
arm's effective training cutoff is roughly 17 months earlier than the default-ES arm's,
against the *same* outer `val_df` — a first-order handicap in exactly the dimension
this project already proved matters most (`notebooks/05_leaderboard_gap_investigation.ipynb`:
validation RMSE nearly doubles in variance terms at long horizons). The observed ~0.8%
degradation is fully consistent with "17 fewer recent months of training data," with no
contribution from the early-stopping mechanism required to explain it. The 5/5-seed
uniformity, which read as "robust" at first glance, is actually consistent with this:
the chronological inner cut is identical in all 5 seeds (only the masking simulation
varies), so a constant confound would produce exactly this uniform pattern — a
seed-varying *mechanism* would not.

No leak was found (every feature in `build_all_features` is strictly backward-looking;
`val_df`'s features come from the same full `fit_raw` in both arms), and the `X_val`/
`y_val` usage matches the sklearn 1.7.2 API correctly.

**A cleaner, confound-free finding was sitting in the results already**:
`current_defaults__default_es` reports `mean_n_iter=300` — exactly `make_baseline_model()`'s
`max_iter` ceiling, in all 5 seeds. Early stopping's internal random holdout **never
actually triggers a stop** for the production hyperparameters — so it cannot have been
responsible for anything, and the original P3 hypothesis is inert for the currently
deployed defaults, without needing the confounded manual-ES experiment to show it. This
also exposes an unrelated, previously-unnoticed cost: `make_baseline_model()` doesn't
pass `early_stopping` explicitly, so it inherits `'auto'`, which turns early stopping ON
for datasets this large (`n_samples > 10_000`) — meaning production has been holding out
10% of its training rows for a stopping check that never fires. `early_stopping=False`
would recover those rows for free with `max_iter=300` unchanged (same ceiling either
way, so no added overfit risk). Tested below.


## Follow-up test 1: does `early_stopping=False` help? (~5 fits)

Recovers the 10% of training rows current production silently discards on a holdout
that (per the finding above) never actually triggers a stop. `max_iter=300` stays
exactly as-is — same ceiling, so this can only add training rows, not remove any
existing regularisation effect (there wasn't one: the holdout never fired).


In [8]:
def make_model_no_early_stopping():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("gbr", HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=300,
            max_depth=8,
            min_samples_leaf=50,
            l2_regularization=1.0,
            early_stopping=False,
            random_state=config.RANDOM_STATE,
        )),
    ])


no_es_rmses = []
for seed in range(5):
    fit_s, val_s = evaluate.mask_augmented_horizon_matched_split(
        raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
        fit_seed=seed, val_seed=seed + 100,
    )
    cols = get_feature_cols(fit_s)
    Xf = features.select_base_features(fit_s, cols)
    yf = fit_s[config.TARGET_COL].to_numpy()
    Xv = features.select_base_features(val_s, cols)
    yv = val_s[config.TARGET_COL].to_numpy()
    m = make_model_no_early_stopping()
    m.fit(Xf, yf)
    no_es_rmses.append(evaluate.rmse(yv, model.predict(m, Xv)))

print(f"early_stopping=False: mean_rmse={np.mean(no_es_rmses):.4f} "
      f"range=({min(no_es_rmses):.4f}, {max(no_es_rmses):.4f})  "
      f"rmses={[round(r, 4) for r in no_es_rmses]}")
print(f"current_defaults__default_es (from confirmation pass above) for comparison: "
      f"mean_rmse={np.mean(confirmation['current_defaults__default_es'][0]):.4f}")


early_stopping=False: mean_rmse=0.7120 range=(0.6952, 0.7311)  rmses=[0.7311, 0.7189, 0.7154, 0.6952, 0.6995]
current_defaults__default_es (from confirmation pass above) for comparison: mean_rmse=0.7117


## Follow-up test 2: did early stopping ever matter for P3's actually-submitted candidate? (1 fit)

Cheap diagnostic, not a full re-evaluation: refit P3's round-2 winning candidate
(`notebooks/10_hyperparameter_tuning.ipynb` — the one that scored 0.758156 real vs. the
current best 0.755129) with `max_iter=1000` and sklearn's DEFAULT early stopping
(exactly as notebook 10 originally ran it), and check `n_iter_`. If it also hit the
1000 ceiling, early stopping was inert for the actually-submitted candidate too, and
P3's real proxy-real inversion needs a different explanation than early stopping —
the leading remaining suspect being a genuine distribution shift between the proxy's
validation window and Test.csv's real 2016-2018 period.


In [9]:
p3_round2_winner = dict(
    learning_rate=0.0039, max_depth=10, min_samples_leaf=63,
    l2_regularization=0.239, max_leaf_nodes=48,
)
fit_s, val_s = evaluate.mask_augmented_horizon_matched_split(
    raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
    fit_seed=0, val_seed=100,
)
cols = get_feature_cols(fit_s)
Xf = features.select_base_features(fit_s, cols)
yf = fit_s[config.TARGET_COL].to_numpy()
Xv = features.select_base_features(val_s, cols)
yv = val_s[config.TARGET_COL].to_numpy()

p3_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gbr", HistGradientBoostingRegressor(
        loss="squared_error", max_iter=1000, random_state=config.RANDOM_STATE,
        **p3_round2_winner,
    )),
])
p3_model.fit(Xf, yf)
p3_n_iter = p3_model.named_steps["gbr"].n_iter_
p3_rmse = evaluate.rmse(yv, model.predict(p3_model, Xv))
print(f"P3 round-2 winner, DEFAULT early stopping: rmse={p3_rmse:.4f} n_iter_={p3_n_iter} "
      f"(ceiling was max_iter=1000)")


P3 round-2 winner, DEFAULT early stopping: rmse=0.7219 n_iter_=1000 (ceiling was max_iter=1000)


## Gate decision — final

- **The original framing was wrong.** "Manual horizon-aware early stopping is
  refuted" does not hold — that comparison was confounded by a chronological inner
  split that deleted the ~17 most recent training months from the manual-ES arm only,
  which alone explains the observed ~0.8% degradation (this project already proved
  long-horizon predictions carry much higher error variance). The correct caption for
  that result is "this particular nested-split procedure loses," not "early stopping
  loses."
- **The early-stopping mechanism is disposed of anyway, cleanly, for both cases that
  matter:**
  - Production defaults: `current_defaults__default_es`'s `mean_n_iter=300` sits
    exactly at the `max_iter` ceiling in all 5 seeds — early stopping never fires.
  - P3's actual submitted candidate: refit with `max_iter=1000` and default early
    stopping, `n_iter_=1000` — also hit its ceiling exactly. **Early stopping did not
    influence the candidate that caused P3's real proxy-real inversion either.** That
    inversion's true cause remains open; the leading remaining suspect is a genuine
    distribution shift between the proxy's validation window and Test.csv's real
    2016-2018 period — not investigated here.
  - Together these are strictly better evidence than the confounded experiment: no
    nested-split artifact, and it covers the one candidate that was actually
    real-world tested.
- **Follow-up test 1 (`early_stopping=False`) is a null result**: mean_rmse=0.7120 vs.
  0.7117 for the current default — indistinguishable, no improvement. Expected, since
  early stopping's holdout was already never triggering a stop (nothing to "recover"
  by disabling it beyond noise-level). **Not graduated, no real submission warranted**
  (nothing beats the current proxy baseline).
- **Nothing in this entire notebook changes `src/model.py`.** Both the original
  manual-ES search (confounded, no candidate beat the baseline anyway) and the
  `early_stopping=False` follow-up are negative results. Logged in
  `README.md`/`RESOURCES.md`.
